# Evaluate Cosmos3-Nano on MVBench

**Download public inputs → validate coverage → infer → score locally.** The default is the fixed five-question Egocentric Navigation smoke, not a full score. Set `RUN_FULL = True` to select full evaluation directly; it does not rerun smoke first.

[MVBench](https://arxiv.org/abs/2311.17005) contains 4,000 annotated multiple-choice questions across 20 video-understanding tasks. This notebook uses the native VLMEvalKit MVBench task definitions, temporal bounds, prompts, and exact-match evaluator. There is no LLM judge or automatic upload. The Nano serving revision is `e59a53c25979a090fa8706c9acc0c254a6e89b92`.

**Research-only media terms:** public access does not establish commercial rights or redistribution permission. The card mentions 320 NTU videos, but this pinned revision's manual list contains 200 RGB AVI files for its 200 Fine-grained Pose questions. Obtain authorized access from https://rose1.ntu.edu.sg/dataset/actionRecognition/ and use the pinned video/MVBench_videos_ntu.txt list. Removed media is never downloaded from historical revisions or mirrors. No published-score reproduction or commercial-rights claim is made. This is not a customer-data evaluation-agent implementation.


## 1 · Choose one execution scope

The external host `MVBENCH_DATA_DIR` is writable at `/mvbench`; caches and results are separate external mounts. No AWS credentials, Docker socket, labels, media, or model weights are embedded in this image.

For **score-only**, choose `SCORE_SCOPE` explicitly (`smoke` or `full`) and set exactly one of `PREDICTIONS_FILE` (native TSV/XLSX/JSON) or `EXISTING_WORK` (a completed controller inference directory). No endpoint check, GPU, model construction, or private S3 access is used. The selected scope must match the predictions exactly; arbitrary partial scores are not accepted.

For full runs, stage authorized NTU files under `/mvbench/video/nturgbd` using Jupyter's authenticated file browser/terminal before preparation. The download cell retrieves only currently approved public files. Preparation reports missing media and refuses to run an incomplete full benchmark.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
from datetime import datetime, timezone
from IPython.display import FileLink, display

SCORE_ONLY = os.environ.get('EVAL_SCORE_ONLY') == '1'
PREDICTIONS_FILE = None
EXISTING_WORK = None
SCORE_SCOPE = 'smoke'  # Explicitly change to 'full' for 4,000 existing predictions.
RUN_FULL = False      # Full inference is opt-in; selecting it skips the smoke.
DOWNLOAD_PUBLIC = True
NFRAME = 16           # Reference frame count; not a model-score parity claim.
API_NPROC = 1
DATA = Path(os.environ.get('MVBENCH_DATA_DIR', '/mvbench'))
SCOPE = SCORE_SCOPE if SCORE_ONLY else ('full' if RUN_FULL else 'smoke')
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK = Path('/results') / f'mvbench-{SCOPE}-{STAMP}'
SCORE_DIR = Path('/results') / f'mvbench-score-{SCOPE}-{STAMP}'
CLI = [sys.executable, '-u', str(Path('/opt/vlmevalkit/launchable/launchable.py'))]

def invoke(command):
    # Stream instead of buffering an entire long-running inference job.
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

if SCOPE not in {'smoke', 'full'}:
    raise ValueError('Choose explicit smoke or full scope.')
if SCORE_ONLY and (PREDICTIONS_FILE is None) == (EXISTING_WORK is None):
    raise ValueError('Set exactly one of PREDICTIONS_FILE or EXISTING_WORK for score-only execution.')
print('Score existing predictions' if SCORE_ONLY else 'Generate and score predictions', SCOPE)
print('Data:', DATA, '| frames:', NFRAME, '| concurrent API workers:', API_NPROC)


## 2 · Download the chosen public scope

The downloader pins `OpenGVLab/MVBench` to `230a2d4fac8900333c61754641c7a13e069ac9c6`, fetches all 20 official annotation JSONs and licensing/NTU notes, and safely extracts selected current public archives, including the June-2024 `data0613` overlay where applicable. Smoke needs only the approximately 40 MB `vlnqa.zip` archive. Full download is substantially larger and **cannot supply the manual NTU videos**. Existing approved files are reused. Set `DOWNLOAD_PUBLIC = False` for offline scoring with already staged inputs.


In [ ]:
if DOWNLOAD_PUBLIC:
    invoke(CLI + ['download', '--data-root', str(DATA), '--scope', SCOPE])
else:
    print('Using already staged public annotations and media.')


## 3 · Prepare trusted annotations and require complete selected coverage

Smoke selects exactly stable indices 3400–3404: five of 4,000 rows, one of 20 tasks. Full requires all 4,000 rows and all 20 tasks; no missing clips or questions are silently dropped. Missing full NTU media is an actionable prerequisite failure, not a partial full score. A structured coverage report is retained under `/mvbench/.mvbench/`.


In [ ]:
scope_flag = ['--smoke'] if SCOPE == 'smoke' else []
invoke(CLI + ['prepare', '--data-root', str(DATA)] + scope_flag)


## 4 · Check the endpoint only for inference

BYOM can be selected before host setup with `COSMOS_API_BASE` (complete `/v1/chat/completions` URL), exact `COSMOS_MODEL`, optional `COSMOS_API_KEY`, and `COSMOS_MODEL_REVISION`. Keep keys in the environment, not this notebook. The endpoint must accept sampled-image messages; fixed frame count is meaningful because no re-encoded video is sent. Generation uses the existing CosmosReason2 recipe: deterministic temperature 0, seed 1, no thinking, maximum 8,192 output tokens.

The default 16 frames and bicubic scale/center-crop to 224 follow the reference recipe. Frames are sent as sampled images; Cosmos owns downstream image encoding/normalization, so VideoChat2 tensor normalization is not applied and model/parser differences remain. There is no paper-score parity claim.


In [ ]:
if not SCORE_ONLY:
    invoke(CLI + ['check', '--wait', '1800'])
else:
    print('Score-only: endpoint check and model inference skipped.')


## 5 · Native inference and local scoring, or score existing outputs

Progress streams here and into the private inference `run.log`. Fresh directories prevent stale prediction reuse. Inference and scoring reject wrong/missing/duplicate identities and blank or transport-failure outputs; the scorer reconstructs trusted identities and labels from pinned annotations. Malformed but nonempty model answers receive native wrong-answer semantics, not fabricated successes. Native exact matching runs locally without an API judge.


In [ ]:
if SCORE_ONLY:
    source = (['--work-dir', str(EXISTING_WORK)] if EXISTING_WORK is not None
              else ['--predictions', str(PREDICTIONS_FILE)])
    invoke(CLI + ['score', '--data-root', str(DATA), '--scope', SCOPE,
                  '--score-dir', str(SCORE_DIR), '--nframe', str(NFRAME)] + source)
    SUMMARY = SCORE_DIR / 'summary.json'
else:
    invoke(CLI + ['run', '--data-root', str(DATA), '--mode', 'all',
                  '--nframe', str(NFRAME), '--api-nproc', str(API_NPROC),
                  '--work-dir', str(WORK)] + scope_flag)
    SUMMARY = WORK / 'scores' / 'summary.json'


## 6 · Inspect metrics, coverage, and provenance

The primary metric is count-based multiple-choice accuracy as a **fraction**, with per-task metrics and the task-macro average reported separately. Smoke is prominently marked as a subset. Native scorer version/source, dataset revision and manifest, prediction hashes, selected IDs, frame protocol, generation parameters, and model/source revisions are recorded in private artifacts. A BYOM display name alone does not establish its checkpoint revision. No reference/published-score parity is claimed until independently established with an agreed protocol.


In [ ]:
summary = json.loads(SUMMARY.read_text())
print(json.dumps({key: summary[key] for key in
                  ('status', 'coverage', 'primary_metric', 'per_task', 'provenance')}, indent=2))
display(FileLink(str(SUMMARY), url_prefix='/files'))
if not SCORE_ONLY:
    invoke(CLI + ['results', '--work-dir', str(WORK)])
    display(FileLink(str(WORK / 'run.json'), url_prefix='/files'))


## Stop and retain private results

Container lifecycle commands belong in the **host** terminal, not this notebook: `docker compose --profile local stop` preserves state. `bash setup.sh --score-only` starts only authenticated Jupyter for existing-prediction scoring. There is no Docker socket in the evaluator and the local model API has no host port. Stopping containers does not stop underlying Brev VM billing; stop that instance separately after retaining authorized results.

Clear outputs before sharing. Do not commit restricted media, labels, credentials, or answer-bearing artifacts. Native Linux x86-64 is the supported runtime; code changes alone do not demonstrate notebook/hardware execution or full benchmark reproduction.
